In [1]:
# ============================================================
# PS S6E5 - exp_20260501_004_lgb_lap_race_time_norm_min
# LightGBM + Race/Driver time normalization minimal safe FE
# ============================================================

In [2]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OrdinalEncoder

import lightgbm as lgb

In [3]:
# ============================================================
# Config
# ============================================================

class CFG:
    EXP_ID = "exp_20260501_004_lgb_lap_race_time_norm_min"
    COMPETITION = "PS S6E5 Predicting F1 Pit Stops"
    METRIC = "AUC"

    BASE = Path("/kaggle/input/competitions/playground-series-s6e5")
    TRAIN_PATH = BASE / "train.csv"
    TEST_PATH = BASE / "test.csv"
    SUB_PATH = BASE / "sample_submission.csv"

    OUTDIR = Path(f"/kaggle/working/{EXP_ID}")

    ID_COL = "id"
    TARGET = "PitNextLap"

    SEED = 42
    N_SPLITS = 5

    PARAMS = {
        "objective": "binary",
        "metric": "auc",
        "boosting_type": "gbdt",
        "learning_rate": 0.03,
        "num_leaves": 64,
        "max_depth": -1,
        "min_child_samples": 40,
        "subsample": 0.85,
        "subsample_freq": 1,
        "colsample_bytree": 0.85,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
        "n_estimators": 5000,
        "random_state": SEED,
        "verbosity": -1,
    }


CFG.OUTDIR.mkdir(parents=True, exist_ok=True)

In [4]:
# ============================================================
# Load data
# ============================================================

train = pd.read_csv(CFG.TRAIN_PATH)
test = pd.read_csv(CFG.TEST_PATH)
sub = pd.read_csv(CFG.SUB_PATH)

print("train shape:", train.shape)
print("test shape :", test.shape)
print("sub shape  :", sub.shape)

print("\ntrain columns:")
print(train.columns.tolist())

print("\ntest columns:")
print(test.columns.tolist())

assert CFG.ID_COL in train.columns
assert CFG.ID_COL in test.columns
assert CFG.TARGET in train.columns
assert CFG.TARGET not in test.columns

feature_cols_raw = [c for c in train.columns if c not in [CFG.ID_COL, CFG.TARGET]]

print("\nraw feature cols:", len(feature_cols_raw))
print(feature_cols_raw)

print("\ntarget distribution:")
print(train[CFG.TARGET].value_counts(dropna=False))
print(train[CFG.TARGET].value_counts(normalize=True, dropna=False))

train shape: (439140, 16)
test shape : (188165, 15)
sub shape  : (188165, 2)

train columns:
['id', 'Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'PitNextLap']

test columns:
['id', 'Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change']

raw feature cols: 14
['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change']

target distribution:
PitNextLap
0.0    351759
1.0     87381
Name: count, dtype: int64
PitNextLap
0.0    0.801018
1.0    0.198982
Name: proportion, dtype: float64


In [5]:
# ============================================================
# Feature Engineering
# ============================================================

def add_group_relative_features(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    """
    Add safe group-relative features using train+test feature distributions.

    This experiment does NOT use target.
    This experiment does NOT use original dataset.
    This experiment does NOT create Normalized_TyreLife proxy.
    This experiment does NOT use Race x LapNumber group statistics.

    Why train+test?
    - These are unsupervised distributional statistics.
    - Same transformation is applied to train/test.
    - This is common in Kaggle tabular FE.
    - Still, we keep it minimal and avoid target/future-like aggregation.
    """

    train_df = train_df.copy()
    test_df = test_df.copy()

    train_df["_is_train"] = 1
    test_df["_is_train"] = 0

    all_df = pd.concat([train_df, test_df], axis=0, ignore_index=True)

    added = []

    def add_group_stats(group_cols, value_col, prefix):
        nonlocal all_df, added

        grp = all_df.groupby(group_cols)[value_col]

        mean_col = f"{prefix}_{value_col}_mean"
        std_col = f"{prefix}_{value_col}_std"
        median_col = f"{prefix}_{value_col}_median"

        all_df[mean_col] = grp.transform("mean")
        all_df[std_col] = grp.transform("std")
        all_df[median_col] = grp.transform("median")

        diff_mean_col = f"{value_col}_minus_{prefix}_mean"
        diff_median_col = f"{value_col}_minus_{prefix}_median"
        z_col = f"{value_col}_z_by_{prefix}"
        ratio_mean_col = f"{value_col}_div_{prefix}_mean"

        all_df[diff_mean_col] = all_df[value_col] - all_df[mean_col]
        all_df[diff_median_col] = all_df[value_col] - all_df[median_col]
        all_df[z_col] = all_df[diff_mean_col] / (all_df[std_col].replace(0, np.nan) + 1e-6)
        all_df[ratio_mean_col] = all_df[value_col] / (all_df[mean_col].replace(0, np.nan) + 1e-6)

        added.extend([
            mean_col,
            std_col,
            median_col,
            diff_mean_col,
            diff_median_col,
            z_col,
            ratio_mean_col,
        ])

    # Race-level normalization:
    # Captures circuit/race scale differences without using target.
    race_value_cols = [
        "LapTime (s)",
        "LapTime_Delta",
        "TyreLife",
        "Cumulative_Degradation",
        "Position",
        "Position_Change",
    ]

    for col in race_value_cols:
        add_group_stats(["Race"], col, "race")

    # Driver x Race-level normalization:
    # Captures each driver's race-specific baseline.
    driver_race_value_cols = [
        "LapTime (s)",
        "LapTime_Delta",
        "TyreLife",
        "Cumulative_Degradation",
        "Position",
        "Position_Change",
    ]

    for col in driver_race_value_cols:
        add_group_stats(["Driver", "Race"], col, "driver_race")

    # Minimal progress/lap interactions.
    # These do not use future stint endpoint.
    all_df["LapNumber_x_RaceProgress"] = all_df["LapNumber"] * all_df["RaceProgress"]
    all_df["TyreLife_x_RaceProgress"] = all_df["TyreLife"] * all_df["RaceProgress"]
    all_df["Cumulative_Degradation_x_RaceProgress"] = all_df["Cumulative_Degradation"] * all_df["RaceProgress"]
    all_df["LapTime_Delta_x_RaceProgress"] = all_df["LapTime_Delta"] * all_df["RaceProgress"]

    added.extend([
        "LapNumber_x_RaceProgress",
        "TyreLife_x_RaceProgress",
        "Cumulative_Degradation_x_RaceProgress",
        "LapTime_Delta_x_RaceProgress",
    ])

    # PitStop interaction:
    # PitStop=1 often means current lap is pit-related. Consecutive pit next lap may be rare.
    all_df["PitStop_x_TyreLife"] = all_df["PitStop"] * all_df["TyreLife"]
    all_df["PitStop_x_RaceProgress"] = all_df["PitStop"] * all_df["RaceProgress"]
    all_df["PitStop_x_LapTime_Delta"] = all_df["PitStop"] * all_df["LapTime_Delta"]

    added.extend([
        "PitStop_x_TyreLife",
        "PitStop_x_RaceProgress",
        "PitStop_x_LapTime_Delta",
    ])

    # Replace infinite values from ratios/z-scores.
    for c in added:
        all_df[c] = all_df[c].replace([np.inf, -np.inf], np.nan)

    train_out = all_df[all_df["_is_train"] == 1].drop(columns=["_is_train"]).reset_index(drop=True)
    test_out = all_df[all_df["_is_train"] == 0].drop(columns=["_is_train"]).reset_index(drop=True)

    # Deduplicate preserving order
    added = list(dict.fromkeys(added))

    return train_out, test_out, added


train_fe, test_fe, added_features = add_group_relative_features(train, test)

feature_cols = [c for c in train_fe.columns if c not in [CFG.ID_COL, CFG.TARGET]]

print("\nadded features:", len(added_features))
for c in added_features:
    print(c)

print("\ntotal feature cols:", len(feature_cols))


added features: 91
race_LapTime (s)_mean
race_LapTime (s)_std
race_LapTime (s)_median
LapTime (s)_minus_race_mean
LapTime (s)_minus_race_median
LapTime (s)_z_by_race
LapTime (s)_div_race_mean
race_LapTime_Delta_mean
race_LapTime_Delta_std
race_LapTime_Delta_median
LapTime_Delta_minus_race_mean
LapTime_Delta_minus_race_median
LapTime_Delta_z_by_race
LapTime_Delta_div_race_mean
race_TyreLife_mean
race_TyreLife_std
race_TyreLife_median
TyreLife_minus_race_mean
TyreLife_minus_race_median
TyreLife_z_by_race
TyreLife_div_race_mean
race_Cumulative_Degradation_mean
race_Cumulative_Degradation_std
race_Cumulative_Degradation_median
Cumulative_Degradation_minus_race_mean
Cumulative_Degradation_minus_race_median
Cumulative_Degradation_z_by_race
Cumulative_Degradation_div_race_mean
race_Position_mean
race_Position_std
race_Position_median
Position_minus_race_mean
Position_minus_race_median
Position_z_by_race
Position_div_race_mean
race_Position_Change_mean
race_Position_Change_std
race_Position_C

In [6]:
# ============================================================
# Column types
# ============================================================

# Keep categorical policy close to previous experiments.
cat_cols = [
    "Driver",
    "Compound",
    "Race",
    "Year",
    "LapNumber",
    "PitStop",
    "Position",
    "Stint",
]
cat_cols = [c for c in cat_cols if c in feature_cols]

num_cols = [c for c in feature_cols if c not in cat_cols]

print("\ncat_cols:", len(cat_cols), cat_cols)
print("num_cols:", len(num_cols), num_cols)


cat_cols: 8 ['Driver', 'Compound', 'Race', 'Year', 'LapNumber', 'PitStop', 'Position', 'Stint']
num_cols: 97 ['TyreLife', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'race_LapTime (s)_mean', 'race_LapTime (s)_std', 'race_LapTime (s)_median', 'LapTime (s)_minus_race_mean', 'LapTime (s)_minus_race_median', 'LapTime (s)_z_by_race', 'LapTime (s)_div_race_mean', 'race_LapTime_Delta_mean', 'race_LapTime_Delta_std', 'race_LapTime_Delta_median', 'LapTime_Delta_minus_race_mean', 'LapTime_Delta_minus_race_median', 'LapTime_Delta_z_by_race', 'LapTime_Delta_div_race_mean', 'race_TyreLife_mean', 'race_TyreLife_std', 'race_TyreLife_median', 'TyreLife_minus_race_mean', 'TyreLife_minus_race_median', 'TyreLife_z_by_race', 'TyreLife_div_race_mean', 'race_Cumulative_Degradation_mean', 'race_Cumulative_Degradation_std', 'race_Cumulative_Degradation_median', 'Cumulative_Degradation_minus_race_mean', 'Cumulative_Degradation_minus_race_median', 'Cumulative_De

In [7]:
# ============================================================
# Prepare X/y
# ============================================================

X = train_fe[feature_cols].copy()
X_test = test_fe[feature_cols].copy()

assert train_fe[CFG.TARGET].notna().all()
assert set(train_fe[CFG.TARGET].dropna().unique()).issubset({0, 1, 0.0, 1.0})
y = train_fe[CFG.TARGET].astype(int).values

X_enc = X.copy()
X_test_enc = X_test.copy()

if cat_cols:
    enc = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1,
        encoded_missing_value=-2,
    )
    X_enc[cat_cols] = enc.fit_transform(
        X_enc[cat_cols].astype("string").fillna("__MISSING__")
    )
    X_test_enc[cat_cols] = enc.transform(
        X_test_enc[cat_cols].astype("string").fillna("__MISSING__")
    )

print("\nX_enc shape:", X_enc.shape)
print("X_test_enc shape:", X_test_enc.shape)

print("\nNaN count in X_enc:", int(X_enc.isna().sum().sum()))
print("NaN count in X_test_enc:", int(X_test_enc.isna().sum().sum()))


X_enc shape: (439140, 105)
X_test_enc shape: (188165, 105)

NaN count in X_enc: 21449
NaN count in X_test_enc: 8986


In [8]:
# ============================================================
# CV Training
# ============================================================

oof = np.zeros(len(train_fe), dtype=float)
pred = np.zeros(len(test_fe), dtype=float)

fold_scores = []
fold_best_iterations = []
feature_importance_list = []

skf = StratifiedKFold(
    n_splits=CFG.N_SPLITS,
    shuffle=True,
    random_state=CFG.SEED,
)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_enc, y), 1):
    print("\n" + "=" * 80)
    print(f"Fold {fold}")
    print("=" * 80)

    X_tr = X_enc.iloc[tr_idx]
    X_va = X_enc.iloc[va_idx]
    y_tr = y[tr_idx]
    y_va = y[va_idx]

    model = lgb.LGBMClassifier(**CFG.PARAMS)

    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(200),
            lgb.log_evaluation(200),
        ],
        categorical_feature=cat_cols if cat_cols else "auto",
    )

    va_pred = model.predict_proba(X_va)[:, 1]
    te_pred = model.predict_proba(X_test_enc)[:, 1]

    oof[va_idx] = va_pred
    pred += te_pred / CFG.N_SPLITS

    auc = roc_auc_score(y_va, va_pred)
    fold_scores.append(float(auc))
    fold_best_iterations.append(int(model.best_iteration_))

    print(f"fold {fold} AUC: {auc:.8f}")
    print(f"fold {fold} best_iteration: {model.best_iteration_}")

    fi = pd.DataFrame({
        "feature": feature_cols,
        "importance": model.feature_importances_,
        "fold": fold,
    })
    feature_importance_list.append(fi)


cv_auc = roc_auc_score(y, oof)

print("\n" + "=" * 80)
print("CV RESULT")
print("=" * 80)
print("CV AUC:", cv_auc)
print("fold_scores:", fold_scores)
print("mean:", np.mean(fold_scores))
print("std :", np.std(fold_scores))
print("fold_best_iterations:", fold_best_iterations)


Fold 1
Training until validation scores don't improve for 200 rounds
[200]	valid_0's auc: 0.939809
[400]	valid_0's auc: 0.941892
[600]	valid_0's auc: 0.942361
[800]	valid_0's auc: 0.942499
Early stopping, best iteration is:
[785]	valid_0's auc: 0.942538
fold 1 AUC: 0.94253848
fold 1 best_iteration: 785

Fold 2
Training until validation scores don't improve for 200 rounds
[200]	valid_0's auc: 0.938171
[400]	valid_0's auc: 0.939966
[600]	valid_0's auc: 0.940532
[800]	valid_0's auc: 0.940857
Early stopping, best iteration is:
[789]	valid_0's auc: 0.940889
fold 2 AUC: 0.94088924
fold 2 best_iteration: 789

Fold 3
Training until validation scores don't improve for 200 rounds
[200]	valid_0's auc: 0.93967
[400]	valid_0's auc: 0.941406
[600]	valid_0's auc: 0.941901
[800]	valid_0's auc: 0.942116
Early stopping, best iteration is:
[769]	valid_0's auc: 0.942161
fold 3 AUC: 0.94216089
fold 3 best_iteration: 769

Fold 4
Training until validation scores don't improve for 200 rounds
[200]	valid_0's 

In [9]:
# ============================================================
# Save artifacts
# ============================================================

np.save(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.npy", oof)
np.save(CFG.OUTDIR / f"pred_{CFG.EXP_ID}.npy", pred)

sub_out = sub.copy()
pred_col = [c for c in sub_out.columns if c != CFG.ID_COL][0]
sub_out[pred_col] = pred
sub_path = CFG.OUTDIR / f"submission_{CFG.EXP_ID}.csv"
sub_out.to_csv(sub_path, index=False)

feature_df = pd.DataFrame({
    "feature": feature_cols,
    "is_added": [c in added_features for c in feature_cols],
    "is_categorical": [c in cat_cols for c in feature_cols],
})
feature_df.to_csv(CFG.OUTDIR / "feature_columns.csv", index=False)

fi_all = pd.concat(feature_importance_list, axis=0, ignore_index=True)
fi_summary = (
    fi_all
    .groupby("feature", as_index=False)["importance"]
    .agg(["mean", "std"])
    .reset_index()
    .sort_values("mean", ascending=False)
)

fi_all.to_csv(CFG.OUTDIR / "feature_importance_by_fold.csv", index=False)
fi_summary.to_csv(CFG.OUTDIR / "feature_importance.csv", index=False)

result = {
    "experiment": {
        "id": CFG.EXP_ID,
        "competition": CFG.COMPETITION,
        "metric": CFG.METRIC,
    },
    "data": {
        "train_shape": list(train.shape),
        "test_shape": list(test.shape),
        "target": CFG.TARGET,
        "id_col": CFG.ID_COL,
        "target_distribution": {
            str(k): int(v)
            for k, v in train[CFG.TARGET].value_counts(dropna=False).to_dict().items()
        },
    },
    "features": {
        "raw_feature_count": len(feature_cols_raw),
        "total_feature_count": len(feature_cols),
        "added_feature_count": len(added_features),
        "categorical_features": cat_cols,
        "numerical_features": num_cols,
        "added_features": added_features,
        "notes": [
            "Race-level and Driver-Race-level unsupervised normalization features are added.",
            "No Race x LapNumber group statistics are used in this minimal version.",
            "No original dataset is used.",
            "No Normalized_TyreLife proxy is used.",
            "No target encoding is used.",
            "No pseudo label is used.",
        ],
    },
    "model": {
        "family": "LightGBM",
        "params": CFG.PARAMS,
        "cv": {
            "scheme": "StratifiedKFold",
            "n_splits": CFG.N_SPLITS,
            "shuffle": True,
            "random_state": CFG.SEED,
        },
    },
    "result": {
        "cv_auc": float(cv_auc),
        "fold_scores": fold_scores,
        "fold_best_iterations": fold_best_iterations,
        "fold_mean": float(np.mean(fold_scores)),
        "fold_std": float(np.std(fold_scores)),
    },
    "artifacts": {
        "oof": str(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.npy"),
        "pred": str(CFG.OUTDIR / f"pred_{CFG.EXP_ID}.npy"),
        "submission": str(sub_path),
        "feature_columns": str(CFG.OUTDIR / "feature_columns.csv"),
        "feature_importance": str(CFG.OUTDIR / "feature_importance.csv"),
    },
}

with open(CFG.OUTDIR / "cv_result.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print("\nSaved artifacts to:", CFG.OUTDIR)
print("Submission:", sub_path)

display(fi_summary.head(50))
display(sub_out.head())


Saved artifacts to: /kaggle/working/exp_20260501_004_lgb_lap_race_time_norm_min
Submission: /kaggle/working/exp_20260501_004_lgb_lap_race_time_norm_min/submission_exp_20260501_004_lgb_lap_race_time_norm_min.csv


,index,feature,mean,std
11,11,Driver,17721.8,615.053006
12,12,LapNumber,7553.8,457.657295
55,55,Race,4490.8,266.843962
37,37,Position,2306.2,215.714395
56,56,RaceProgress,654.2,37.452637
58,58,TyreLife,633.4,28.693205
68,68,Year,511.6,22.941229
23,23,LapTime_Delta,494.6,25.667100
57,57,Stint,429.0,9.874209
60,60,TyreLife_div_race_mean,407.2,23.794957


,id,PitNextLap
0,439140,0.004710
1,439141,0.001855
2,439142,0.007851
3,439143,0.151877
4,439144,0.871695
